In [ ]:
import pandas as pd
import numpy as np
import json
import os
import ast
from pathlib import Path
import torch
from typing import List, Optional
from dataclasses import dataclass

import teradatasql
from sqlalchemy import text, create_engine
from teradataml import create_context, get_context, get_connection, DataFrame, in_schema, copy_to_sql
from teradataml.dataframe.copy_to import copy_to_sql
from dotenv import load_dotenv

# import sys
# sys.path.append('..')
from constants import (
    CLEANED_TEST_DATA_PATH,
    ENCODED_TEST_DATA_PATH,
    CLEANED_TRAIN_DATA_PATH
)

In [ ]:
# load_dotenv('../.env')

# TD_HOST = os.getenv('TD_HOST')
# TD_USER = os.getenv('TD_USER')
# TD_PASS = os.getenv('TD_PASS')
# TD_DB = os.getenv('TD_DB')

TD_HOST="iteration7-w9og53takluu3v27.env.clearscape.teradata.com"
TD_USER="demo_user"
TD_PASS="n8888888"
TD_DB="DEMO_USER"

In [ ]:
# print(TD_DB,TD_HOST,TD_PASS,TD_USER)
# conn = teradatasql.connect(
#     host=TD_HOST,
#     user=TD_USER,
#     password=TD_PASS,
#     database=TD_DB
# )
# cursor = conn.cursor()
# print("Successfully connected to Teradata!")

In [ ]:
conn = teradatasql.connect(
    host=TD_HOST,
    user=TD_USER,
    password=TD_PASS,
    logdata={'CHARSET': 'UTF8'}
)

sqlalchemy_engine = create_engine("teradatasql://", creator=lambda: conn)
create_context(tdsqlengine=sqlalchemy_engine)
print("Connection successful with UTF-8 encoding!")

In [ ]:
tables_df = DataFrame.from_query(f"""
    SELECT DatabaseName, TableName
    FROM DBC.TablesV
    WHERE DatabaseName = '{TD_DB}'
""")

tables_df

In [ ]:
tdf_labels = DataFrame.from_table("original_labels_fc", schema_name=TD_DB, index_label="row_id")
print("Shape of the data:", tdf_labels.shape)
tdf_labels.head(5)

In [ ]:
tdf = DataFrame.from_table("original_dataset", schema_name=TD_DB, index_label="Item_Name")
print("Shape of the data:", tdf.shape)

In [ ]:
tdf.head(10)

In [ ]:
tdf.tdtypes

In [ ]:
tdf = tdf.dropna(subset=["Item_Name", "class"])

In [ ]:
tdf.count()

In [ ]:
tdf = tdf.assign(Item_Name = tdf.Item_Name.str.lower())
tdf_stripped = tdf.assign(Item_Name = tdf.Item_Name.str.strip())

In [ ]:
tdf = tdf.assign(
    Item_Name = tdf.Item_Name.otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", "")
)

In [ ]:
cleaned_tdf = tdf[["Item_Name", "class"]]
cleaned_tdf

In [ ]:
from teradataml import execute_sql
from teradatasqlalchemy.types import VARCHAR
unicode_type = VARCHAR(length=500, charset='UNICODE')

unicode_tdf = cleaned_tdf.assign(
    Item_Name = cleaned_tdf.Item_Name.cast(unicode_type)
)

source_query = unicode_tdf.show_query()

create_sql = f"""
CREATE TABLE {target_table_name} AS (
    {source_query}
) WITH DATA;
"""

print("--- Generated SQL ---")
print(create_sql)

try:
    print(f"\nAttempting to drop existing table '{target_table_name}'...")
    execute_sql(f"DROP TABLE {target_table_name};")
    print("✔ Previous table dropped.")
except Exception:
    print("Table did not exist, proceeding to create.")

print("\nExecuting CREATE TABLE statement... 🚀")
execute_sql(create_sql)
print(f"✔ Success! Table '{target_table_name}' has been created.")

In [ ]:
tdf_2 = DataFrame.from_table("cleaned_data", schema_name=TD_DB)
print("Shape of the data:", tdf_labels.shape)
tdf_2.head(5)

In [ ]:
tables_df = DataFrame.from_query(f"""
    SELECT DatabaseName, TableName
    FROM DBC.TablesV
    WHERE DatabaseName = '{TD_DB}'
""")

tables_df

In [ ]:
## Verify unicode on teradata session

# td_context = get_context()
# raw_connection = td_context.engine.raw_connection()
# print("Verifying the character set of the established session...")
# with raw_connection.cursor() as cur:
#     cur.execute("HELP SESSION;")
    
#     session_info = cur.fetchone()
    
#     if session_info:
#         character_set = session_info[5]
#         print(f"✅ Session Character Set is: {character_set}")
#     else:
#         print("❌ Could not retrieve session information.")

# raw_connection.close()

In [ ]:
# copy_to_sql(
#     df=cleaned_tdf,              
#     table_name="cleaned_data",
#     if_exists="replace"
# )

In [ ]:
# tdf_2 =  tdf[['Item_Name', 'class']]
# tdf_2
######
# cat=tdf.groupby(['class']).count()
# cat
#######

In [ ]:
# remove_context()